In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from model import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [51]:
df = pd.read_csv("cardio_cleaned.csv")
df.head()

,age_group,height_group,weight_group,ap_hi_group,ap_lo_group,gender,cholesterol,gluc,smoke,alco,active,cardio
0,3,1,2,2,1,1,2,2,0,0,1,0
1,0,2,3,1,1,1,1,1,0,0,1,1
2,2,2,2,1,1,1,1,1,0,0,1,0
3,0,2,4,1,1,2,1,1,1,1,1,0
4,3,1,2,1,1,1,1,1,0,0,1,0


In [52]:
y = df["cardio"]
X = df.drop(columns=["cardio"])

In [ ]:
depths = list(range(1,21))          # testowane głębokości 1..20
seeds  = list(range(10))            # 10 różnych random_state

In [54]:
rows = []  # zbierzemy tu wyniki z każdej kombinacji (seed, depth)

for seed in seeds:
    # 70/15/15 z zachowaniem proporcji klas
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
    )
    
    for d in depths:
        clf = DecisionTreeClassifier(max_depth=d)
        clf.fit(X_train.values.tolist(), y_train.values.tolist())

        acc_tr = accuracy_score(y_train, clf.predict(X_train.values.tolist()))
        acc_val = accuracy_score(y_val, clf.predict(X_val.values.tolist()))

        rows.append({
            "seed": seed,
            "max_depth": d,
            "train_acc": acc_tr,
            "val_acc": acc_val
        })

results = pd.DataFrame(rows)

In [55]:
# Średnie i odchylenia po seedach
summary = (
    results.groupby("max_depth")[["train_acc", "val_acc"]]
    .agg(["mean", "std"])
)
# Porządniejsze nazwy kolumn
summary.columns = ["train_mean", "train_std", "val_mean", "val_std"]
summary = summary.reset_index()

display(summary)

,max_depth,train_mean,train_std,val_mean,val_std
0,1,0.710232,0.000581,0.709406,0.001804
1,2,0.717810,0.000663,0.716310,0.003350
2,3,0.727508,0.000998,0.724653,0.003454
3,4,0.732850,0.001247,0.725618,0.003349
4,5,0.740672,0.001157,0.721353,0.003630
5,6,0.752171,0.001034,0.714163,0.004189
6,7,0.765417,0.000756,0.707387,0.005519
7,8,0.776918,0.000668,0.702502,0.005584
8,9,0.785268,0.000582,0.699626,0.004223
9,10,0.789926,0.000522,0.697242,0.004625


In [56]:
# Wybór najlepszej głębokości po ŚREDNIM train accuracy (remis -> mniejsze depth)
best_row = summary.sort_values(
    by=["train_mean", "max_depth"], ascending=[False, True]
).iloc[0]
best_depth = int(best_row["max_depth"])
print(f"Najlepsza głębokość po średnim TRAIN accuracy: {best_depth} "
      f"(train_mean={best_row['train_mean']:.4f}, val_mean={best_row['val_mean']:.4f})")

Najlepsza głębokość po średnim TRAIN accuracy: 11 (train_mean=0.7917, val_mean=0.6962)


In [57]:
# Wybór najlepszej głębokości po ŚREDNIM val accuracy (remis -> mniejsze depth)
best_row = summary.sort_values(
    by=["val_mean", "max_depth"], ascending=[False, True]
).iloc[0]
best_depth = int(best_row["max_depth"])
print(f"Najlepsza głębokość po średnim VAL accuracy: {best_depth} "
      f"(val_mean={best_row['val_mean']:.4f}, train_mean={best_row['train_mean']:.4f})")

Najlepsza głębokość po średnim VAL accuracy: 4 (val_mean=0.7256, train_mean=0.7328)


In [ ]:
# Dla wybranej głębokości (na podstawie walidacji) licze średni TEST accuracy (po seedach)
test_accs = []

for seed in seeds:
    # 70/15/15
    X_train, X_temp, y_train, y_temp = train_test_split(X.values.tolist(), y.values.tolist(),test_size=0.30, random_state=seed, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp,  y_temp, test_size=0.50, random_state=seed, stratify=y_temp)

    clf = DecisionTreeClassifier(max_depth=best_depth)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    test_accs.append(accuracy_score(y_test, y_pred))

print(f"TEST (best depth={best_depth}): {np.mean(test_accs):.4f} ± {np.std(test_accs):.4f}")

TEST (best depth=4): 0.7274 ± 0.0034
